In [1]:
import chromadb
from chromadb.utils import embedding_functions

PydanticImportError: `BaseSettings` has been moved to the `pydantic-settings` package. See https://docs.pydantic.dev/2.12/migration/#basesettings-has-moved-to-pydantic-settings for more details.

For further information visit https://errors.pydantic.dev/2.12/u/import-error

In [5]:

!pip install sentence-transformers

  Using cached transformers-5.0.0-py3-none-any.whl.metadata (37 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached transformers-5.0.0-py3-none-any.whl (10.1 MB)
Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl (447 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 MB 39.4 MB/s  0:00:02m0:00:0100:01
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 46.5 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.0/28.0 MB 43.4 MB/s  0:00:00m0:00:0100:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [sentence-transformers]ence-transformers]


In [2]:
import chromadb
from chromadb.utils import embedding_functions

In [6]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name = "all-MiniLM-L6-V2"
)

/Users/anirudh/.local/share/virtualenvs/Desktop-sFnGVMJ4/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█| 103/103 [00:00<00:00, 926.91it/s, Materializing param=poole
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-V2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
client = chromadb.Client()
my_collections = "my_grocery_collection"


In [30]:
# Define the main function to interact with the Chroma DB
def main():
    try:
        # Place your database operations inside this block
        # Create a collection in the Chroma database with a specified name, 
        # distance metric, and embedding function. In this case, we are using 
        # cosine distance
        collection = client.get_or_create_collection(
        name = my_collections,
        metadata={"description": "A collection for storing grocery data"},
        configuration ={
            "hnsw":{"space":"cosine"},
            "embedding_function":ef
        }    
    
)
        print(f"Collection created: {collection.name}")
        # Array of grocery-related text items
        texts = [
            'fresh red apples',
            'organic bananas',
            'ripe mangoes',
            'whole wheat bread',
            'farm-fresh eggs',
            'natural yogurt',
            'frozen vegetables',
            'grass-fed beef',
            'free-range chicken',
            'fresh salmon fillet',
            'aromatic coffee beans',
            'pure honey',
            'golden apple',
            'red fruit'
        ]
        ids = [f"food_{index+1}" for index, _ in enumerate(texts)]
        collection.add(
            documents = texts,
            metadatas=[{"source": "grocery_store", "category": "food"} for _ in texts],
            ids = ids
        )
        all_items = collection.get()
        print("Collection contnts:")
        print(f"Number of documents is {len(all_items['documents'])}")

        def perform_similarity_search(collection, all_items):
            try:
                query_term = ["red","fresh"]
                results = collection.query(
                    query_texts = query_term,
                    n_results = 3
                )
                print(f"Query results for {query_term}")
                print(results)
                if not results or not results['ids'] or len(results['ids'][0]) == 0:
                    print(f"No documents similar to {query_term} exist")
                    return
                for i in range (min(3,len(results['ids'][0]))):
                    doc_id = results['ids'][0][i]
                    score = results['distances'][0][i]
                    text = results ['documents'][0][i]
                    if not text:
                        print(f' - ID: {doc_id}, Text: "Text not available", Score: {score:.4f}')
                    else:
                        print(f' - ID: {doc_id}, Text: "{text}", Score: {score:.4f}')
                    
            except Exception as error:
                print(f"Error in similarity search: {error}")
                
        perform_similarity_search(collection,all_items)
    except Exception as error:  # Catch any errors and log them to the console
            print(f"Error: {error}")
main()        
        

Collection created: my_grocery_collection
Collection contnts:
Number of documents is 14
Query results for ['red', 'fresh']
{'ids': [['food_14', 'food_1', 'food_13'], ['food_1', 'food_5', 'food_12']], 'embeddings': None, 'documents': [['red fruit', 'fresh red apples', 'golden apple'], ['fresh red apples', 'farm-fresh eggs', 'pure honey']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'category': 'food', 'source': 'grocery_store'}, {'source': 'grocery_store', 'category': 'food'}, {'source': 'grocery_store', 'category': 'food'}], [{'category': 'food', 'source': 'grocery_store'}, {'category': 'food', 'source': 'grocery_store'}, {'category': 'food', 'source': 'grocery_store'}]], 'distances': [[0.31327712535858154, 0.45399630069732666, 0.7393020391464233], [0.477375864982605, 0.4854103922843933, 0.6252565979957581]]}
 - ID: food_14, Text: "red fruit", Score: 0.3133
 - ID: food_1, Text: "fresh red apples", Score: 0.4540
 - ID: food_13, Text:

In [ ]:
# 1. Setup
import chromadb
from chromadb.utils import embedding_functions
# 2. Create embedding function and client
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.Client()
# 3. Create collection with configuration
collection = client.create_collection(
    name="collection_name",
    configuration={"hnsw": {"space": "cosine"}, "embedding_function": ef}
)
# 4. Add documents
collection.add(documents=texts, metadatas=metadata, ids=ids)
# 5. Perform similarity search
results = collection.query(query_texts=["query"], n_results=5)
# 6. Process results
for i, (doc_id, score, text) in enumerate(zip(results['ids'][0], results['distances'][0], results['documents'][0])):
    print(f"Rank {i+1}: {doc_id}, Score: {score:.4f}, Text: {text}")